# Week 5 · 06 PyTorch basics: everything from 01–05, packaged

PyTorch is what you've built this week, made fast and convenient: tensors are NumPy arrays that can also track gradients and live on a GPU (`01`), autograd is `04`'s engine for whole arrays, `nn.Linear` is `02`'s dense layer, `torch.optim` is `05`. This notebook starts from a trained, saved digit classifier (§E), then goes through each piece, the five-line training loop, data loading, a full training run on the 8×8 digits, and MNIST. §10 is code-reading practice for the exam. Two days: §M–7, then §8–10.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **tensor**: PyTorch's array: like a NumPy array, plus an optional gradient record and a device (CPU or GPU).
- **dtype**: the number type: `torch.float32` (PyTorch's default for decimals), `torch.float64` (NumPy's default), `torch.int64` (labels).
- **device**: where a tensor lives and is computed: `"cpu"`, `"cuda"` (an NVIDIA GPU) or `"mps"` (a Mac's GPU).
- **`nn.Module`**: the base class of every layer and model: holds parameters, and `model(x)` runs its `forward`.
- **state_dict**: a dictionary from parameter names to tensors: everything a trained model stores.
- **`Dataset` / `DataLoader`**: `ds[i]` gives sample $i$; a `DataLoader` shuffles and stacks samples into batches.
- **logits**: a classifier's raw scores, before softmax. PyTorch's classification loss takes these.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from figures import *
import checks; checks.load(globals(), "06")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** Every box is something you built by hand earlier this week; here you meet PyTorch's version of it.

In [ ]:
fig_torch_map()

| Section | What | Why it's here | You built it in |
|---|---|---|---|
| 0 | recap, and the E-step from memory | the vectorized line from `01`, a week later |  |
| E | a saved digit classifier predicts a new image | start from the end: what a trained PyTorch model stores and does | `02` §E |
| 1 | tensors | NumPy with two extras; the dtype and memory traps | `01` |
| 2 | autograd | `.grad`, accumulation, `no_grad` | `04` |
| 3 | `nn.Linear` and `nn.Module` | layers, models, `(out, in)` weights | `02` |
| 4 | losses | `CrossEntropyLoss` takes logits and integer labels | `02` §6–7 |
| 5 | the training loop | five lines, as an algorithm card | `03`, `05` |
| 6 | `Dataset` and `DataLoader` | batches, the way PyTorch does them | `05` §2 |
| 7 | `train()`, `eval()`, `no_grad` | the evaluation pattern |  |
| 8 | digits, end to end | train, evaluate, save, load: the code that made §E's model |  |
| 9 | MNIST, and the GPU | the real 28×28 dataset; `.to(device)` |  |
| 10 | reading PyTorch code | exam practice: shapes, bugs, outputs |  |

## 0. Recap first · 10 min, closed book

**Quick check:** `nn.Linear(784, 128)`: the shape of its weight?

<details><summary>Answer</summary>

`(128, 784)`: PyTorch stores (out, in). It computes `x @ W.T + b`. [02 §E ↗](02_neural_networks.ipynb)

</details>

**Quick check:** Why must gradients be zeroed before every `backward()`?

<details><summary>Answer</summary>

`backward()` **adds** into `.grad` (a value can feed several operations, so contributions must add up). Without zeroing, the previous step's gradients are added again. [04 §3 ↗](04_micrograd.ipynb)

</details>

**Quick check:** `softmax` + cross-entropy: $\partial L / \partial z = $ ?

<details><summary>Answer</summary>

$p - e_y$ (divided by $n$ for a batch mean). [02 §7 ↗](02_neural_networks.ipynb)

</details>

**Quick check:** Adam's update in one line, and its default learning rate?

<details><summary>Answer</summary>

$w \leftarrow w - \eta\, \hat m / (\sqrt{\hat s} + \varepsilon)$ with bias-corrected averages of $g$ and $g^2$; $\eta = 10^{-3}$. [05 §5 ↗](05_optimizers.ipynb)

</details>

**Quick check:** An epoch with $n = 1000$ samples and batches of 64: how many steps?

<details><summary>Answer</summary>

$\lceil 1000 / 64 \rceil = 16$; the last batch has $1000 - 15 \cdot 64 = 40$ samples. [05 §2 ↗](05_optimizers.ipynb)

</details>

**Exercise 1 · The E-step, from memory.** $r_{ik} = \dfrac{\pi_k\, \mathcal{N}(x_i \mid \mu_k, \sigma_k^2)}{\sum_j \pi_j\, \mathcal{N}(x_i \mid \mu_j, \sigma_j^2)}$; `x` `(n,)`; `mu`, `var`, `pi` `(K,)`. Return **shape `(n, K)`**, no loops. `gaussian_pdf` is given in the cell.

**Example**: what goes in, and what should come out:
```python
x = np.array([1, 2, 3, 6, 7, 8.0])
e_step(x, np.array([2.0, 5.0]), np.array([4.0, 4.0]), np.array([0.5, 0.5]))[:, 0]    # → [0.867 0.755 0.593 0.133 0.068 0.033]
```

In [ ]:
def gaussian_pdf(x, mu, var):
    return np.exp(-(x - mu) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var)

def e_step(x, mu, var, pi):
    pass  # your code

In [ ]:
check(1)

## E. Start from the end: a saved model reads a new digit

`digits_mlp.pt` in this folder is a trained classifier for 8×8 images of handwritten digits (scikit-learn's digits dataset). It's a two-layer network, $64 \to 32 \to 10$, trained by the code in §8. A saved model is just its **state_dict**: names and tensors.

In [ ]:
model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))   # the same shapes it was saved with
model.load_state_dict(torch.load("digits_mlp.pt"))                       # copy the stored numbers in
model.eval()                                                             # prediction mode (§7)
for name, t in model.state_dict().items():
    print(name, tuple(t.shape))
# → 0.weight (32, 64)
#   0.bias (32,)
#   2.weight (10, 32)
#   2.bias (10,)

| Stored | Shape | What it is | Built in |
|---|---|---|---|
| `0.weight` | (32, 64) | layer 0 (`nn.Linear(64, 32)`): **row** $k$ = hidden unit $k$'s weights on the 64 pixels | §8 (training) |
| `0.bias` | (32,) | one bias per hidden unit | §8 |
| `2.weight` | (10, 32) | layer 2: row $c$ = digit $c$'s weights on the 32 hidden units (layer 1 is the ReLU, which stores nothing) | §8 |
| `2.bias` | (10,) | one bias per digit | §8 |

$64 \cdot 32 + 32 + 32 \cdot 10 + 10 = 2410$ numbers in all. **A new image** (from the test set, never used in training) goes in as a row of 64 pixel values in $[0, 1]$:

In [ ]:
X_tr, X_te, y_tr, y_te = digits_data()              # from figures.py: (1347, 64), (450, 64), (1347,), (450,)
x = torch.tensor(X_te[:1], dtype=torch.float32)       # (1, 64): one image as a batch of size 1
with torch.no_grad():                                 # predicting only: don't record gradients
    logits = model(x)                                 # (1, 10)
    probs = torch.softmax(logits, dim=1)              # (1, 10), the row sums to 1
print(y_te[0], probs.argmax(dim=1).item(), round(probs.max().item(), 3))   # → 2 2 0.977

🖼 **Picture.** The image and the model's probabilities: 0.977 for "2", with small amounts on "3" and "0".

In [ ]:
fig_digit_new()

`model(x)` is nothing more than `02`'s forward pass, with PyTorch's (out, in) weights:

In [ ]:
W0, b0, W2, b2 = (model.state_dict()[k] for k in ["0.weight", "0.bias", "2.weight", "2.bias"])
by_hand = torch.relu(x @ W0.T + b0) @ W2.T + b2      # (1, 64) @ (64, 32) -> (1, 32) -> (1, 10)
print(torch.allclose(by_hand, logits))                # → True

**The rest of this notebook answers:** what are these objects (tensors, `nn.Linear`, `nn.Sequential`, a state_dict), and what code turned random numbers into these 2410 trained ones (§5, §8)?

## 1. Tensors: NumPy with two extras

A tensor works like a NumPy array (indexing, broadcasting, `@`, `.sum(dim=...)`) and adds two things: it can record the operations done to it, for gradients (§2), and it can live on a GPU (§9). The names differ a little:

| NumPy | PyTorch | Note |
|---|---|---|
| `np.array([...])` | `torch.tensor([...])` | copies the data |
| `a.shape` | `t.shape` (or `t.size()`) | a `torch.Size`, which behaves like a tuple |
| `axis=`, `keepdims=` | `dim=`, `keepdim=` | same meaning |
| `a.astype(np.float32)` | `t.float()` or `t.to(torch.float32)` |  |
| `np.concatenate`, `np.stack` | `torch.cat`, `torch.stack` |  |
| `a.reshape(...)` | `t.reshape(...)` or `t.view(...)` | `view` never copies (it needs contiguous memory) |
| `np.random.default_rng(0).normal(size=...)` | `torch.manual_seed(0); torch.randn(...)` | PyTorch has one global random generator |
| `torch.from_numpy(a)` / `t.numpy()` |  | convert both ways, **sharing** memory (CPU only) |
| `a.item()` | `t.item()` | a one-element tensor → a Python number |

**dtypes: the most common error.** PyTorch's default for decimals is float32 (GPUs are fast at it); NumPy's is float64. Labels for classification must be int64:

In [ ]:
a = torch.tensor([1, 2, 3])                     # whole numbers -> int64
b = torch.tensor([1.0, 2.0, 3.0])               # decimals -> float32
n = np.array([1.0, 2.0, 3.0])                   # NumPy: float64
print(a.dtype, b.dtype, torch.tensor(n).dtype, torch.from_numpy(n).dtype)
# → torch.int64 torch.float32 torch.float64 torch.float64

In [ ]:
W64 = torch.tensor(np.ones((3, 2)))             # float64, because it came from NumPy
x32 = torch.ones(4, 3)                          # float32, PyTorch's default
try:
    x32 @ W64
except RuntimeError as e:
    print("RuntimeError:", e)
# → RuntimeError: expected m1 and m2 to have the same dtype, but got: float != double
print((x32 @ W64.float()).shape)                # → torch.Size([4, 2])   fixed by converting one side

Fix it at the source: build tensors from NumPy with `torch.tensor(X, dtype=torch.float32)`, as §E did.

**Shared memory.** `torch.from_numpy(a)` doesn't copy: the tensor and the array are two views of the same numbers, so changing one changes the other (the `00` A2 lesson again). `torch.tensor(a)` copies.

In [ ]:
arr = np.zeros(3)
t = torch.from_numpy(arr)          # shares memory with arr
arr[0] = 5.0
print(t)                           # → tensor([5., 0., 0.], dtype=torch.float64)

**Dimensions** work exactly like NumPy axes:

In [ ]:
T = torch.arange(6.0).reshape(2, 3)           # like np.arange(6.).reshape(2, 3)
print(T.sum(dim=0), T.sum(dim=1, keepdim=True).shape, T.argmax(dim=1))
# → tensor([3., 5., 7.]) torch.Size([2, 1]) tensor([2, 2])

**Exercise 2 · From NumPy to tensors.** Return `(X_t, y_t)`: `X` `(n, d)` as a **float32** tensor and `y` `(n,)` as an **int64** tensor (what `nn.CrossEntropyLoss` needs).

**Example**: what goes in, and what should come out:
```python
Xt, yt = to_tensors(np.array([[0.5, 1.0], [2.0, 3.0]]), np.array([1, 0]))
Xt.dtype, yt.dtype    # → (torch.float32, torch.int64)
```

In [ ]:
def to_tensors(X, y):
    pass  # your code

In [ ]:
check(2)

## 2. Autograd: `04`'s engine, for arrays

A tensor created with `requires_grad=True` is a **leaf** that PyTorch tracks. Every operation on it records its backward rule (like your `_backward`), and `loss.backward()` runs them all in reverse topological order, adding into each leaf's `.grad`. Example: $L = (w \cdot x - 1)^2$ with $w = (1, -2)$, $x = (3, 0.5)$. Then $w \cdot x = 2$, $L = 1$, and $\nabla_w L = 2(w \cdot x - 1)\, x = (6, 1)$.

In [ ]:
w = torch.tensor([1.0, -2.0], requires_grad=True)     # a parameter: track it
x = torch.tensor([3.0, 0.5])                           # data: not tracked
loss = (w @ x - 1) ** 2
loss.backward()
print(loss.item(), w.grad)                             # → 1.0 tensor([6., 1.])
print([type(f).__name__ for f in (loss.grad_fn, loss.grad_fn.next_functions[0][0], loss.grad_fn.next_functions[0][0].next_functions[0][0])])
# → ['PowBackward0', 'SubBackward0', 'DotBackward0']

`grad_fn` is your `_backward` plus `_prev`: the recorded operation, linked to the ones before it (here: power ← subtraction ← dot product). **Gradients accumulate**, exactly as in your engine:

In [ ]:
loss = (w @ x - 1) ** 2            # a new forward pass
loss.backward()                    # without zeroing first
print(w.grad)                      # → tensor([12.,  2.])   6 + 6 and 1 + 1: added to the old gradient
w.grad = None                      # what optimizer.zero_grad() does (it sets every .grad to None)

**When not to record.** Predicting and evaluating need no gradients; `with torch.no_grad():` switches recording off (less memory, faster). `t.detach()` gives the same numbers without the record; `.numpy()` refuses tensors that are being tracked:

In [ ]:
with torch.no_grad():
    y_pred = w * 2
print(y_pred.requires_grad, (w * 2).requires_grad)     # → False True
try:
    w.numpy()
except RuntimeError as e:
    print("RuntimeError:", e)
# → RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.
print(w.detach().numpy())                              # → [ 1. -2.]

## 3. `nn.Linear` and `nn.Module`

`nn.Linear(in_features, out_features)` is `02`'s dense layer. It creates its own weight and bias (as tracked tensors called **parameters**), with small random starting values (uniform in $\pm 1/\sqrt{\text{in}}$).

| Attribute | Shape | A row is | Computes |
|---|---|---|---|
| `lin.weight` | (out, in) | one output unit's weights | `lin(x) = x @ lin.weight.T + lin.bias` |
| `lin.bias` | (out,) |  | for `x` of shape `(n, in)`, the output is `(n, out)` |

**Why (out, in)?** It's the math convention $y = Wx + b$ for a single column vector $x$, where row $k$ of $W$ is unit $k$'s weights. With samples as rows, that becomes `x @ W.T`. So when copying your NumPy `W` `(in, out)` into PyTorch, transpose it, as `02` §10 did.

In [ ]:
torch.manual_seed(0)
lin = nn.Linear(3, 2)                                  # 3 inputs -> 2 outputs
print(lin.weight.shape, lin.bias.shape)                # → torch.Size([2, 3]) torch.Size([2])
xb = torch.randn(4, 3)                                 # a batch of 4 samples
print(torch.allclose(lin(xb), xb @ lin.weight.T + lin.bias))   # → True

**A model is a class.** Subclass `nn.Module`; in `__init__`, call `super().__init__()` first, then assign layers as attributes (that **registers** their parameters); in `forward`, say what happens to an input. Calling `model(x)` runs `forward` (through `__call__`, `04` §1a). `parameters()` gives every registered parameter, which is what you hand to the optimizer.

In [ ]:
class TwoLayer(nn.Module):
    def __init__(self, d_in, hidden, d_out):
        super().__init__()                      # sets up nn.Module's bookkeeping: must come first
        self.fc1 = nn.Linear(d_in, hidden)      # assigning a layer as an attribute registers its parameters
        self.fc2 = nn.Linear(hidden, d_out)

    def forward(self, x):                       # model(x) runs this
        return self.fc2(torch.relu(self.fc1(x)))

net = TwoLayer(64, 32, 10)
print([(name, tuple(p.shape)) for name, p in net.named_parameters()])
# → [('fc1.weight', (32, 64)), ('fc1.bias', (32,)), ('fc2.weight', (10, 32)), ('fc2.bias', (10,))]
print(sum(p.numel() for p in net.parameters()))   # → 2410

> ⚠️ **Common mistake.** Forgetting `super().__init__()`. The first layer assignment then fails with `AttributeError: cannot assign module before Module.__init__() call`.

For a plain stack of layers, `nn.Sequential(layer1, layer2, ...)` saves writing the class; its layers are named `0`, `1`, `2`, ..., which is where §E's names `0.weight` and `2.weight` come from.

**Exercise 3 · A `nn.Linear` from NumPy weights.** `W` is a NumPy array `(in, out)` (our convention), `b` is `(out,)`. Return an `nn.Linear(in, out)` whose output equals `X @ W + b`. Set the numbers inside `with torch.no_grad():` using `lin.weight.copy_(...)` (the trailing `_` means "in place"), and remember the transpose and float32.

**Example**: what goes in, and what should come out:
```python
lin = linear_from_numpy(np.array([[1.0, 0.0, -1.0], [2.0, 1.0, 0.0]]), np.zeros(3))   # (2, 3): 2 inputs -> 3 outputs
lin.weight.shape                          # → torch.Size([3, 2])
lin(torch.tensor([[1.0, 2.0]]))           # → tensor([[ 5.,  2., -1.]], grad_fn=<AddmmBackward0>)
```

In [ ]:
def linear_from_numpy(W, b):
    pass  # your code

In [ ]:
check(3)

**Exercise 4 · An MLP from a list of sizes.** Return an `nn.Sequential` of `nn.Linear(sizes[0], sizes[1])`, `nn.ReLU()`, ..., ending with the last `nn.Linear` (**no** activation after it: the outputs are logits). Hint: build a Python list of layers, then `nn.Sequential(*layers)` (the `*` passes the list's items as separate arguments).

**Example**: what goes in, and what should come out:
```python
make_mlp([64, 32, 10])
# → Sequential(
#      (0): Linear(in_features=64, out_features=32, bias=True)
#      (1): ReLU()
#      (2): Linear(in_features=32, out_features=10, bias=True)
#    )
```

In [ ]:
def make_mlp(sizes):
    pass  # your code

In [ ]:
check(4)

## 4. Losses: `CrossEntropyLoss` takes logits and integer labels

| Loss | Model output | Target | Use for |
|---|---|---|---|
| `nn.CrossEntropyLoss()` | logits `(n, C)`, **no softmax** | class indices `(n,)`, int64 | classification with $C$ classes |
| `nn.BCEWithLogitsLoss()` | one logit per sample `(n,)` or `(n, 1)` | 0/1 as **float**, same shape | two classes |
| `nn.MSELoss()` | any shape | same shape, float | regression |

**Proposition.** For logits $Z$ $(n, C)$ and labels $y$, `nn.CrossEntropyLoss()(Z, y)` $= \frac{1}{n}\sum_i \left(-z_{i, y_i} + \log \sum_c e^{z_{ic}}\right)$, the mean cross-entropy of the softmax probabilities.

**Proof.** $-\log \operatorname{softmax}(z_i)_{y_i} = -\log \frac{e^{z_{i,y_i}}}{\sum_c e^{z_{ic}}} = -z_{i, y_i} + \log \sum_c e^{z_{ic}}$ (`02` §7), averaged over the batch (the default `reduction="mean"`). $\blacksquare$

**Remarks.** (1) The $\log \sum e$ part is computed stably inside (`torch.logsumexp`), so you never need a softmax for training. (2) Its gradient with respect to $Z$ is $\frac{1}{n}(P - Y)$, so `loss.backward()` starts from exactly `02`'s formula. **Example, checked two ways:** row 1 is `02` §7's example ($0.417$), row 2 has equal logits ($\log 3 = 1.099$); the mean is $0.758$:

In [ ]:
loss_fn = nn.CrossEntropyLoss()
Z = torch.tensor([[2.0, 1.0, 0.1], [0.0, 0.0, 0.0]])        # logits (2, 3)
y = torch.tensor([0, 2])                                    # int64 labels (2,)
by_hand = (-Z[torch.arange(2), y] + torch.logsumexp(Z, dim=1)).mean()
print(round(loss_fn(Z, y).item(), 4), round(by_hand.item(), 4))      # → 0.7578 0.7578

**Two classic bugs.** (1) Applying softmax first: the loss then softmaxes **again**, so the probabilities it scores are much flatter and the gradients are wrong; it still trains, just badly, so nothing warns you. (2) Float labels:

In [ ]:
print(round(loss_fn(torch.softmax(Z, dim=1), y).item(), 4))   # → 0.9504   not 0.7578: a double softmax
try:
    loss_fn(Z, y.float())
except RuntimeError as e:
    print("RuntimeError:", e)
# → RuntimeError: expected target dtype to be Long or Byte, but got Float

## 5. The training loop: an algorithm card

**Goal.** Change the model's parameters to make the mean loss on the training data small.

**Steps.** Before the loop: a model, a loss function, and an optimizer given `model.parameters()`. Then for each epoch: call `model.train()`, and for each batch `(xb, yb)` from the `DataLoader` (step $t$):

In [ ]:
# The shape of every training loop (comments only: model, loader, loss_fn and opt come in §8)
# for epoch in range(num_epochs):
#     model.train()
#     for xb, yb in loader:
#         logits = model(xb)              # 1. forward                       (02)
#         loss = loss_fn(logits, yb)      # 2. loss, a single number         (02 §6)
#         opt.zero_grad()                 # 3. clear the old gradients       (04 §3: they accumulate)
#         loss.backward()                 # 4. every parameter's .grad       (03, 04)
#         opt.step()                      # 5. update the parameters         (05)
#     # after each epoch: evaluate with model.eval() and torch.no_grad() (§7)

**Why it works.** Each step is one minibatch SGD step (`05` §2): steps 1–4 compute an unbiased estimate of the gradient, step 5 moves against it with the optimizer's rule.

**Caveats.** (1) **Order**: `zero_grad()` must come before `backward()` (it can also go first in the loop body); forget it and every step adds the previous gradients again. (2) Log the loss with `loss.item()`, not `loss`: keeping the tensor keeps its whole graph in memory. (3) `model.train()` and `model.eval()` matter as soon as the model has dropout or batch normalization (Week 6). (4) Data, model and labels must have matching dtypes and be on the same device (§9).

**Cost.** One forward and one backward pass per batch: about three forward passes' worth of computation.

**Example trace: `03`'s training step in PyTorch.** Same network, same numbers; one pass through steps 1–5 gives the same new weights as `03` §E's table:

In [ ]:
net = nn.Sequential(nn.Linear(2, 2), nn.Sigmoid(), nn.Linear(2, 1), nn.Sigmoid()).double()   # .double(): float64
with torch.no_grad():
    net[0].weight.copy_(torch.tensor([[0.1, 0.2], [0.3, -0.1]]))   # (out, in): 03's W1 (in, out), transposed
    net[0].bias.copy_(torch.tensor([0.0, 0.1]))
    net[2].weight.copy_(torch.tensor([[0.4, -0.5]]))
    net[2].bias.copy_(torch.tensor([0.2]))
opt = torch.optim.SGD(net.parameters(), lr=0.5)
loss_fn_b = nn.BCELoss()                   # binary cross-entropy on probabilities (this net ends with a sigmoid)
xb = torch.tensor([[1.0, 2.0]], dtype=torch.float64); yb = torch.tensor([[1.0]], dtype=torch.float64)

loss = loss_fn_b(net(xb), yb)              # 1 + 2
opt.zero_grad()                            # 3
loss.backward()                            # 4
opt.step()                                 # 5
print(round(loss.item(), 4), net[0].weight.detach().numpy().T.round(4).tolist())
# → 0.6099 [[0.1215, 0.2717], [0.2429, -0.1565]]

(In real code, drop the final `nn.Sigmoid()` and use `nn.BCEWithLogitsLoss()`: the same loss, computed more stably from the logit.)

## 6. `Dataset` and `DataLoader`

`05` §2 cut a shuffled permutation into minibatches by hand. PyTorch splits this into two objects: a **`Dataset`** knows how many samples there are and how to get sample $i$; a **`DataLoader`** shuffles the indices each epoch and stacks the samples of each batch into tensors.

🖼 **Picture.** An illustration with 10 samples and `batch_size=4`: three batches per epoch, the last with 2 samples (`drop_last=True` would skip it).

In [ ]:
fig_dataloader()

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, Dataset
Xs = torch.arange(20.0).reshape(10, 2); ys = torch.arange(10)          # 10 samples, 2 features each
ds = TensorDataset(Xs, ys)                                             # pairs up the rows: ds[i] = (Xs[i], ys[i])
print(len(ds), ds[3])                                                  # → 10 (tensor([6., 7.]), tensor(3))
loader = DataLoader(ds, batch_size=4, shuffle=True, generator=torch.Generator().manual_seed(0))   # generator: reproducible shuffling
for xb, yb in loader:
    print(tuple(xb.shape), yb.tolist())
# → (4, 2) [3, 7, 5, 2]
#   (4, 2) [0, 8, 1, 6]
#   (2, 2) [9, 4]

**Your own `Dataset`** (for data that doesn't fit in one tensor, e.g. image files on disk) is a class with `__len__` (so `len(ds)` works) and `__getitem__` (so `ds[i]` works), the special methods of `04` §1a:

In [ ]:
class Squares(Dataset):
    def __init__(self, n):
        self.n = n
    def __len__(self):                    # len(ds)
        return self.n
    def __getitem__(self, i):             # ds[i]: build sample i on demand
        x = torch.tensor([float(i)])
        return x, x ** 2

sq = Squares(5)
print(len(sq), sq[3])                                       # → 5 (tensor([3.]), tensor([9.]))
print(next(iter(DataLoader(sq, batch_size=5)))[1].ravel())  # → tensor([ 0.,  1.,  4.,  9., 16.])

## 7. `train()`, `eval()` and `torch.no_grad()`

**The question:** why does evaluation code always call `model.eval()` and wrap everything in `torch.no_grad()`? They do two different things.

|  | What it changes | Why |
|---|---|---|
| `model.train()` / `model.eval()` | the behaviour of layers such as dropout and batch normalization | dropout randomly zeros units while training only; batch norm uses batch statistics while training, stored ones when evaluating (Week 6) |
| `with torch.no_grad():` | whether operations are recorded for `backward()` | evaluation never calls `backward()`, so recording only wastes memory and time |

For the MLPs in this notebook `eval()` changes nothing, but call it anyway: the habit matters as soon as a model has dropout. Here is dropout ($p = 0.5$) in both modes:

In [ ]:
drop = nn.Dropout(p=0.5)
torch.manual_seed(0)
ones = torch.ones(8)
drop.train(); print(drop(ones))     # → tensor([0., 0., 2., 0., 0., 0., 2., 2.])   zeros about half, doubles the rest
drop.eval();  print(drop(ones))     # → tensor([1., 1., 1., 1., 1., 1., 1., 1.])   does nothing

(The kept values are doubled so that the **expected** output stays 1, the same as in evaluation mode.)

**Recipe (evaluating a classifier).** `model.eval()`; inside `torch.no_grad()`, loop over the test loader; count `(model(xb).argmax(dim=1) == yb).sum().item()`; divide by `len(loader.dataset)`. Call `model.train()` again before more training.

**Exercise 5 · One training epoch.** Call `model.train()`, then run the five-line loop over `loader`. Return the **mean loss per sample** over the epoch: add up `loss.item() * len(xb)` for each batch and divide by `len(loader.dataset)` (batches can differ in size).

**Example**: what goes in, and what should come out:
```python
# on the digits, in §8:
train_epoch(model, train_loader, nn.CrossEntropyLoss(), opt)    # → about 1.8 for the first epoch from random weights
```

In [ ]:
def train_epoch(model, loader, loss_fn, opt):
    pass  # your code

In [ ]:
check(5)

**Exercise 6 · Accuracy on a loader.** Follow the recipe above. Return the fraction of samples whose largest logit is the true class, a float in $[0, 1]$. The check verifies that the model is in evaluation mode afterwards.

**Example**: what goes in, and what should come out:
```python
evaluate(model, test_loader)    # → 0.978 for §E's trained digits model
```

In [ ]:
def evaluate(model, loader):
    pass  # your code

In [ ]:
check(6)

## 8. Digits, end to end

Everything together on scikit-learn's 8×8 digits: 1797 images, 64 pixels each (scaled to $[0, 1]$), 10 classes; 75% for training, 25% for testing. This is exactly the code that produced `digits_mlp.pt` (the seed makes it reproducible).

In [ ]:
X_tr, X_te, y_tr, y_te = digits_data()                    # NumPy: (1347, 64), (450, 64), (1347,), (450,)
X_tr_t, y_tr_t = torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr)
X_te_t, y_te_t = torch.tensor(X_te, dtype=torch.float32), torch.tensor(y_te)

torch.manual_seed(0)                                       # same starting weights and shuffles every run
model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=256)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(1, 31):
    model.train()
    for xb, yb in train_loader:
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    if epoch in (1, 10, 20, 30):
        model.eval()
        with torch.no_grad():
            acc = (model(X_te_t).argmax(dim=1) == y_te_t).float().mean().item()
        print(epoch, round(acc, 3))
# → 1 0.82
#   10 0.956
#   20 0.964
#   30 0.978

🖼 **Picture.** The same run, every epoch: the training loss falls steadily; test accuracy jumps in the first few epochs and then creeps up to 0.978.

In [ ]:
fig_digits_training()

**Saving and loading.** Save the **state_dict** (the numbers), not the whole object. To load, build a model with the same shapes, then copy the numbers in:

In [ ]:
torch.save(model.state_dict(), "my_digits_mlp.pt")                       # a dict of tensors, written to a file
fresh = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))     # same shapes, random numbers
fresh.load_state_dict(torch.load("my_digits_mlp.pt"))                     # the saved numbers, copied in
fresh.eval()
with torch.no_grad():
    print(torch.equal(fresh(X_te_t).argmax(1), model(X_te_t).argmax(1)))  # → True

**Exercise 7 · A model class for the digits.** `DigitsMLP(d_in=64, hidden=32, n_classes=10)`, a subclass of `nn.Module`: `nn.Linear(d_in, hidden)`, ReLU, `nn.Linear(hidden, n_classes)`; `forward` returns the logits. Create the first layer before the second (the check reads `parameters()` in order).

**Example**: what goes in, and what should come out:
```python
net7 = DigitsMLP()
sum(p.numel() for p in net7.parameters())    # → 2410
net7(torch.zeros(5, 64)).shape              # → torch.Size([5, 10])
```

In [ ]:
class DigitsMLP(nn.Module):
    def __init__(self, d_in=64, hidden=32, n_classes=10):
        pass  # your code

    def forward(self, x):
        pass  # your code

In [ ]:
check(7)

Your class, trained with your `train_epoch` and checked with your `evaluate` (run after exercises 5–7):

In [ ]:
torch.manual_seed(0)
mine = DigitsMLP()
opt7 = torch.optim.Adam(mine.parameters(), lr=0.01)
for epoch in range(30):
    train_epoch(mine, train_loader, nn.CrossEntropyLoss(), opt7)
print(evaluate(mine, test_loader) > 0.95)      # → True

## 9. MNIST, and the GPU

MNIST: 70,000 handwritten digits, 28×28 pixels (60,000 for training, 10,000 for testing). `torchvision` downloads it the first time (about 65 MB, into `week5/data/`, which git ignores). `transforms.ToTensor()` turns each image into a float32 tensor of shape `(1, 28, 28)` (1 = one colour channel) with values in $[0, 1]$.

In [ ]:
from torchvision import datasets, transforms
train_set = datasets.MNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
test_set = datasets.MNIST(root="data", train=False, download=True, transform=transforms.ToTensor())
img, label = train_set[0]
print(len(train_set), len(test_set), tuple(img.shape), img.dtype, label)   # → 60000 10000 (1, 28, 28) torch.float32 5

**Devices.** A model and its data must be on the same device. `.to(device)` moves a model's parameters (in place) or returns a moved copy of a tensor. On this Mac the GPU is `"mps"`; on a lab server it would be `"cuda"`.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
mnist_model = nn.Sequential(nn.Flatten(),                      # (B, 1, 28, 28) -> (B, 784)
                            nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
opt_m = torch.optim.Adam(mnist_model.parameters(), lr=1e-3)
train_dl = DataLoader(train_set, batch_size=128, shuffle=True)
test_dl = DataLoader(test_set, batch_size=1000)

mnist_model.train()
for xb, yb in train_dl:                                         # ONE epoch: 469 steps
    xb, yb = xb.to(device), yb.to(device)                       # move each batch to the model's device
    loss = loss_fn(mnist_model(xb), yb)
    opt_m.zero_grad(); loss.backward(); opt_m.step()

mnist_model.eval(); correct = 0
with torch.no_grad():
    for xb, yb in test_dl:
        correct += (mnist_model(xb.to(device)).argmax(dim=1).cpu() == yb).sum().item()   # .cpu(): back to compare with yb
print(f"device {device}: test accuracy after one epoch {correct / len(test_set):.3f}")

Expect roughly 0.93–0.96 after one epoch (0.935 on this Mac's GPU when the notebook was built; the exact number varies by device). More epochs, and the convolutional networks of Week 6, get past 0.99.

## 10. Reading PyTorch code: exam drills

On paper first: the lab exam asks you to read and interpret PyTorch code. Each drill names the skill it practises.

**Drill (on paper).** **Shapes.** Give the shape after each lettered line.\n```python\nx = torch.randn(32, 1, 28, 28)        # a batch of MNIST images\nh = x.view(x.size(0), -1)             # (a)\nh = nn.Linear(784, 256)(h)            # (b)\nh = torch.relu(h)\nout = nn.Linear(256, 10)(h)           # (c)\np = out.softmax(dim=1)                # (d)\npred = p.argmax(dim=1)                # (e)\n```

<details><summary>Worked answer</summary>

(a) `(32, 784)`: `view` keeps 32 rows, and `-1` means "whatever makes the sizes fit", $1 \cdot 28 \cdot 28 = 784$. (b) `(32, 256)`. (c) `(32, 10)`. (d) `(32, 10)`, each row summing to 1. (e) `(32,)`, int64: one predicted class per image.

</details>

**Drill (on paper).** **Find the three bugs.**\n```python\nmodel = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10), nn.Softmax(dim=1))\nloss_fn = nn.CrossEntropyLoss()\nopt = torch.optim.Adam(model.parameters(), lr=1e-3)\nfor xb, yb in loader:\n    out = model(xb.view(-1, 784))\n    loss = loss_fn(out, yb.float())\n    loss.backward()\n    opt.step()\n```

<details><summary>Worked answer</summary>

(1) `nn.Softmax` at the end: `CrossEntropyLoss` expects logits, so this is a double softmax (§4). Remove it. (2) `yb.float()`: the labels must be int64 class indices; remove `.float()`. (3) No `opt.zero_grad()`: gradients accumulate across batches. Add it before `loss.backward()`. (Also good practice: `model.train()` before the loop.)

</details>

**Drill (on paper).** **What does it print?**\n```python\nlin = nn.Linear(784, 128)\nprint(lin.weight.shape, lin.bias.shape, sum(p.numel() for p in lin.parameters()))\n```

<details><summary>Worked answer</summary>

`torch.Size([128, 784]) torch.Size([128]) 100480`: weight is (out, in); $784 \cdot 128 + 128 = 100{,}480$.

</details>

**Drill (on paper).** **What does it print?**\n```python\nw = torch.tensor(2.0, requires_grad=True)\nfor _ in range(3):\n    loss = w ** 2\n    loss.backward()\nprint(w.grad)\n```

<details><summary>Worked answer</summary>

`tensor(12.)`. Each `backward()` adds $\frac{d}{dw} w^2 = 2w = 4$, and nothing zeroes it or changes $w$: $4 + 4 + 4 = 12$.

</details>

**Drill (on paper).** **What does this module compute, what shapes go in and out, and how many parameters does it have?**\n```python\nclass Block(nn.Module):\n    def __init__(self, d):\n        super().__init__()\n        self.fc1 = nn.Linear(d, d)\n        self.fc2 = nn.Linear(d, d)\n    def forward(self, x):\n        return x + self.fc2(torch.relu(self.fc1(x)))\n```

<details><summary>Worked answer</summary>

Input `(n, d)`, output `(n, d)`: the input must have the same shape as the output, because they are added. It computes $x + F(x)$ with $F$ a two-layer MLP: a **residual** (skip) connection, the building block of ResNets (Week 7). Parameters: $2(d^2 + d)$.

</details>

**Drill (on paper).** **Why the last line?**\n```python\nmodel.eval()\nwith torch.no_grad():\n    acc = evaluate_on(test_loader)\nmodel.train()\n```

<details><summary>Worked answer</summary>

To switch dropout and batch normalization back to training behaviour before the next training epoch. `no_grad` ends with its `with` block on its own, but `eval()` stays on until you undo it.

</details>

**Recipe (reading any PyTorch model).** (1) In `__init__`, list every layer with its (in, out) sizes. (2) In `forward`, follow one input and write the shape after every line. (3) Find where the logits come out, and check that the loss matches them (logits for `CrossEntropyLoss`). (4) In the loop, check the five lines and their order.

In [ ]:
for n in range(1, 8):
    check(n)

---
Solutions: `solutions/06_solutions.py`. Next: `07_paper_practice.ipynb`, exam-style practice for the whole week.